In [51]:
from Bio.PDB import PDBParser, MMCIFParser, Superimposer
from scipy.spatial import distance
from pymol import cmd
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

In [35]:
def get_first_atom_coordinates(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("structure", pdb_file)
    atom = [i for i in structure.get_atoms()][0]
    return atom.get_coord()

def get_alignment_matrix_in_pymol(path_ref, path_mob):
    cmd.reinitialize()
    cmd.load(path_ref, "ref")
    cmd.load(path_mob, "mob")   
    cmd.align("mob and name CA", "ref and name CA")
    matrix = cmd.get_object_matrix("mob")
    matrix = np.array(matrix).reshape(4, 4)
    return matrix

def transform_coordinate(matrix, coordinate):
    rotation = matrix[:3, :3]
    translation = matrix[:3, 3]
    transformed_coord = np.dot(rotation, coordinate) + translation
    return transformed_coord

In [36]:
# Define paths
root = "."
detected_pockets_dir = os.path.abspath(os.path.join(root, "..", "processed", "detected_pockets"))
raw_alphafill_dir = os.path.abspath(os.path.join(root, "..", "data", "structures", 'alphafill_database'))
aligned_dir = os.path.abspath(os.path.join(root, "..", "processed", "aligned_relaxed_structures"))

# Pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))

# Get list of proteins
proteins = sorted(os.listdir(detected_pockets_dir))

In [57]:
RESULTS = []

# For each protein
for protein in tqdm(proteins):

    # Path to AlphaFill
    path_to_alphafill = os.path.join(raw_alphafill_dir, protein, f"{protein}.cif")

    # Get HETATM atoms
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("alphafill", path_to_alphafill)
    het_atoms = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.id[0].startswith("H_"):
                    for atom in residue:
                        het_atoms.append(atom)

    # Get HETATM resiudes
    residues = set([i.get_parent().get_resname() for i in het_atoms])

    # Get all structures
    sts = sorted(os.listdir(os.path.join(aligned_dir, protein)))

    # For each structure
    for st in sts:

        # Get alignment matrix to AlphaFill structure
        path_to_st = os.path.join(aligned_dir, protein, st)
        matrix = get_alignment_matrix_in_pymol(path_to_alphafill, path_to_st)

        # For each detected pocket
        pockets = pocket_detection_data[pocket_detection_data['File name'] == st]['Pocket number'].tolist()
        for pocket in pockets:
            
            # Get path to pocket
            path_to_pocket = os.path.join(detected_pockets_dir, protein, st.replace('.pdb', ''), 'pockets', f"pocket_{pocket}.pdb")

            # Get 3D coordinate
            coord = get_first_atom_coordinates(path_to_pocket)

            # Apply transformation
            transformed_coord = transform_coordinate(matrix, coord)

            # Calculate distances against HETATMs
            distances = [[i.fullname, i.get_parent().get_resname(), round(distance.euclidean(i.get_coord(), transformed_coord), 3)] for i in het_atoms]
            distances = sorted(distances, key=lambda x: x[2])[0][1:]

            RESULTS.append([protein, st.replace(".pdb", "", pocket), pocket] + distances)

            # Save transformed PDB
            # x, y, z = transformed_coord
            # with open(f"/home/acomajuncosa/Documents/tmp/pocket_{pocket}.pdb", "w") as pdb_file:
            #     pdb_file.write(f"HEADER    Pocket {pocket} Centroid\n")
            #     pdb_file.write(
            #         f"HETATM{pocket:5d} C   LIG A{pocket:4d}    "
            #         f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00          C\n"
            #     )
            #     pdb_file.write("END\n")

RESULTS = pd.DataFrame(RESULTS, columns=['protein', 'structure', 'pocket', 'AlphaFill closest ligand', 'distance'])


 24%|██▍       | 5/21 [00:06<00:19,  1.23s/it]

 Executive-Detail: _struct_conn name lookup failed: AF/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AG/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AH/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AF/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AG/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AH/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AF/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AG/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AH/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AF/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AG/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AH/PYZ/1/N2/ W/NAD/0/C4N/
 Executive-Detail: _struct_conn name lookup failed: AF/PYZ/1/N2/

100%|██████████| 21/21 [00:30<00:00,  1.46s/it]


In [58]:
RESULTS

,protein,structure,pocket,AlphaFill closest ligand,distance
0,P9WFS9,alphafold2_P9WFS9_model_0,1,ATP,1.202
1,P9WFS9,alphafold2_P9WFS9_model_0,2,TSB,0.536
2,P9WFS9,alphafold2_P9WFS9_model_0,3,TAR,9.871
3,P9WFS9,alphafold2_P9WFS9_model_0,4,TAR,3.228
4,P9WFS9,alphafold2_P9WFS9_model_0,5,TSB,14.515
...,...,...,...,...,...
271,P9WQA1,chai1_P9WQA1_model_4,1,TAR,0.921
272,P9WQA1,chai1_P9WQA1_model_4,2,GJY,9.831
273,P9WQA1,swissmodel_P9WQA1_model_0,1,GJY,8.889
274,P9WQA1,swissmodel_P9WQA1_model_0,2,UNU,0.465
